# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Pick a baseline and up to 4 variant scenarios below — every chart and table in
this notebook updates automatically as the selection changes, no re-running
required. Each must already have a run (`analysis sim load <name> && analysis
sim run <name>`); the newest run for each is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).



In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, display

from analysis.sim.scenario_compare import available_scenarios, scenario_picker_widgets

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", REPO / "data" / "shared" / "sim"))
MAX_VARIANTS = 4

_available = available_scenarios(DATA_DIR)
_default_baseline = "current" if "current" in _available else _available[0]
_default_variants = [name for name in _available if name != _default_baseline][:MAX_VARIANTS]

baseline_picker, variant_picker = scenario_picker_widgets(
    _available, _default_baseline, _default_variants
)
display(widgets.VBox([baseline_picker, variant_picker]))


## Are these runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
from analysis.sim.scenario_compare import display_warning, render_load_or_warn

_META = [
    "scenario_name", "status", "finished_at",
    "code_version", "code_dirty", "data_version", "data_dirty",
]


def _render_comparability(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return

    meta = pd.concat(
        [result.runs[name].items("simulation_run")[_META] for name in result.scenarios],
        ignore_index=True,
    )
    for column in ("code_version", "data_version"):
        meta[column] = meta[column].str[:12]
    display(meta.set_index("scenario_name").T)

    warnings = []
    if meta["data_version"].nunique() > 1:
        warnings.append("different **data versions** — the data repo changed between runs")
    if meta["code_version"].nunique() > 1:
        warnings.append("different **code versions** — the engine changed between runs")
    if meta[["code_dirty", "data_dirty"]].to_numpy().any():
        warnings.append("at least one run was made from **uncommitted** code or data")
    if (meta["status"] != "succeeded").any():
        warnings.append("at least one run did **not succeed**")
    if warnings:
        for warning in warnings:
            display_warning(warning)
    else:
        display(Markdown("> ✅ Same code and data version; differences below are the scenarios."))


display(widgets.interactive_output(
    _render_comparability, {"baseline": baseline_picker, "variants": variant_picker}
))


## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
from analysis.sim.scenario_compare import display_warning

_COMPARE = ["amount_cents", "start", "end", "series", "cadence", "tax_character"]


def _render_rules_and_events(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return
    runs, scenarios = result.runs, result.scenarios
    base, variants = result.baseline, result.variants

    rules = {
        name: runs[name].items("scheduled").set_index("name").reindex(columns=_COMPARE)
        for name in scenarios
    }
    _index = sorted(set().union(*(frame.index for frame in rules.values())))
    rules = {name: frame.reindex(_index) for name, frame in rules.items()}

    _base = rules[base]
    _differs = pd.DataFrame(False, index=_index, columns=_COMPARE)
    for name in variants:
        other = rules[name]
        _differs |= (_base != other) & ~(_base.isna() & other.isna())

    _rows = [
        {"rule": rule, "field": field,
         **{name: rules[name].at[rule, field] for name in scenarios}}
        for rule in _index if _differs.loc[rule].any()
        for field in _COMPARE if _differs.at[rule, field]
    ]
    differences = pd.DataFrame(_rows)
    if differences.empty:
        display_warning("**All scenarios have identical rules.**")
    else:
        display(differences.set_index(["rule", "field"]).fillna("—"))

    events = pd.concat(
        [runs[name].items("calendar_event").assign(scenario=name) for name in scenarios],
        ignore_index=True,
    )
    if events.empty:
        display(Markdown("_No calendar events in any scenario._"))
    else:
        events["amount"] = (events["amount"].astype("Float64") / 100).map(
            lambda v: "" if pd.isna(v) else f"${v:,.2f}"
        )
        # Drop events whose (date, kind, scheduled_name, amount, description) is
        # identical across every scenario -- only differences are worth showing.
        _sig_cols = ["date", "kind", "scheduled_name", "amount", "description"]
        _sig = events[_sig_cols].apply(tuple, axis=1)
        _sig_scenario_count = events.groupby(_sig)["scenario"].transform("nunique")
        events = events[_sig_scenario_count < len(scenarios)]

        if events.empty:
            display_warning("**All scenarios have identical calendar events.**")
        else:
            display(
                events[["scenario", "date", "kind", "scheduled_name", "amount", "description"]]
                .set_index(["scenario", "date"])
            )


display(widgets.interactive_output(
    _render_rules_and_events, {"baseline": baseline_picker, "variants": variant_picker}
))


## Net worth over time

Nominal dollars. (`net_worth_constant_cpi` is available in each run if you want
the inflation-adjusted view later.)


In [ ]:
from analysis.sim.scenario_compare import net_worth_by_month, plot_net_worth_and_delta, variant_delta


def _render_net_worth(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return
    nw = net_worth_by_month(result.runs, result.scenarios)
    delta = variant_delta(nw, result.baseline, result.variants)
    plot_net_worth_and_delta(nw, delta, result.baseline, result.variants, result.colors)


display(widgets.interactive_output(
    _render_net_worth, {"baseline": baseline_picker, "variants": variant_picker}
))


## Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
from analysis.sim.scenario_compare import breakeven_summary


def _render_breakeven(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return
    nw = net_worth_by_month(result.runs, result.scenarios)
    delta = variant_delta(nw, result.baseline, result.variants)
    summary = breakeven_summary(nw, delta, result.baseline, result.variants)
    display(
        summary.style.format({
            "deepest deficit": "${:,.0f}",
            "at scenario end": "${:,.0f}",
            "final net worth": "${:,.0f}",
        })
    )
    display(Markdown(
        f"`{result.baseline}` final net worth: **${nw[result.baseline].iloc[-1]:,.0f}**"
    ))


display(widgets.interactive_output(
    _render_breakeven, {"baseline": baseline_picker, "variants": variant_picker}
))


## Horizon sensitivity — how much does this matter, and by when?

A claim-timing decision only pays off if you live long enough to collect on it, so
the honest question isn't "what's the difference at 2060" but "what's the
difference **if we stop looking at year X**". Every point on these curves answers
that: the gap at year X *is* the impact of the decision if the money stops
mattering then.

The two panels are deliberately on different scales, because they answer different
questions:

- **The decision** — what waiting is worth. Small, slow, and it starts negative.
- **The risk** — what a Trust Fund shortfall costs. Large, and unrelated to the
  claim date.

If the decision is worth little at your realistic horizons, then taking the
benefit earlier for certainty and convenience costs you very little.


In [ ]:
from analysis.sim.scenario_compare import (
    comparison_frame,
    filter_comparisons,
    has_any_comparisons,
    plot_horizon_comparisons,
)

# The pairs worth contrasting: (scenario, its counterfactual). Edit for other questions.
COMPARISONS = {
    "the decision": {
        "wait — no cut": ("january", "current"),
        "wait — with cut": ("january-cut", "current-cut"),
    },
    "the risk": {
        "cut — claiming now": ("current-cut", "current"),
        "cut — waiting": ("january-cut", "january"),
    },
}


def _render_horizon_chart(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return
    comparisons = filter_comparisons(COMPARISONS, result.scenarios)
    if not has_any_comparisons(comparisons):
        display(Markdown(
            "_None of the configured `COMPARISONS` pairs apply to this selection "
            "— pick scenarios from `COMPARISONS`, or edit it above to add a pair "
            "for this selection._"
        ))
        return
    nw = net_worth_by_month(result.runs, result.scenarios)
    compare = comparison_frame(nw, comparisons)
    plot_horizon_comparisons(compare, comparisons)


display(widgets.interactive_output(
    _render_horizon_chart, {"baseline": baseline_picker, "variants": variant_picker}
))


In [ ]:
from analysis.sim.scenario_compare import outstanding_tax_by_year, year_end_values


def _render_horizon_detail(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return

    comparisons = filter_comparisons(COMPARISONS, result.scenarios)
    if not has_any_comparisons(comparisons):
        display(Markdown(
            "_None of the configured `COMPARISONS` pairs apply to this selection "
            "— see the horizon-sensitivity chart above._"
        ))
        return

    nw = net_worth_by_month(result.runs, result.scenarios)
    compare = comparison_frame(nw, comparisons)

    # Year-end value of each comparison, and the pot it is a share of.
    horizon = year_end_values(compare)
    horizon.index.name = "if we look out to"
    pot = year_end_values(nw[result.baseline])

    def _horizon_table(years, frame):
        table = frame.loc[years].copy()
        for label in compare.columns:
            table[f"{label} (% of pot)"] = 100 * frame.loc[years, label] / pot.loc[years]
        return table.style.format(
            {c: ("{:+.2f}%" if c.endswith("(% of pot)") else "${:+,.0f}")
             for c in table.columns}
        )

    outstanding = pd.DataFrame(
        {name: outstanding_tax_by_year(result.runs, name) for name in result.scenarios}
    ).fillna(0)
    nw_year_end = year_end_values(nw)
    settled = nw_year_end.sub(outstanding.reindex(nw_year_end.index).fillna(0), axis=0)
    compare_settled = comparison_frame(settled, comparisons)
    compare_settled.index.name = "if we look out to"

    shift = (compare_settled - horizon).abs().max(axis=1)
    moved = shift[shift >= 1].index
    display(Markdown(
        f"Settling changes the answer in **{len(moved)} of {len(horizon)}** years"
        + (f" ({', '.join(str(y) for y in moved)}):" if len(moved) else ".")
    ))
    if len(moved):
        display(
            pd.concat(
                {"as-of": horizon.loc[moved], "settled": compare_settled.loc[moved]},
                axis=1,
            ).style.format("${:+,.0f}")
        )

    marks = list(range(horizon.index.min() + 4, horizon.index.max() + 1, 5))
    display(Markdown("**At five-year horizons** (tax owed settled):"))
    display(_horizon_table(marks, compare_settled))

    def _at_year(year, settle_tax=True):
        """Net worth already reflects tax paid; `settle_tax` also clears what is owed."""
        frame = compare_settled if settle_tax else horizon
        basis = "after settling the year's tax bill" if settle_tax else "as of Dec 31"
        lines = [
            f"**If the money stops mattering at the end of {year}** "
            f"({basis}; net worth {'$'}{pot.loc[year]:,.0f}):",
            "",
        ]
        lines += [
            f"- {label}: **${value:+,.0f}** ({100 * value / pot.loc[year]:+.2f}% of the pot)"
            for label, value in frame.loc[year].items()
        ]
        display(Markdown("\n".join(lines)))

    year_slider = widgets.IntSlider(
        min=int(horizon.index.min()), max=int(horizon.index.max()),
        value=min(2036, int(horizon.index.max())), description="horizon",
        continuous_update=False,
    )
    settle_checkbox = widgets.Checkbox(value=True, description="settle tax owed")
    display(widgets.VBox([
        widgets.HBox([year_slider, settle_checkbox]),
        widgets.interactive_output(_at_year, {"year": year_slider, "settle_tax": settle_checkbox}),
    ]))


display(widgets.interactive_output(
    _render_horizon_detail, {"baseline": baseline_picker, "variants": variant_picker}
))


### Settling the tax bill at the horizon

Net worth above is after-tax — the engine pays tax as a real transfer — but a tax
year settles the *following* April. So at the end of year X each scenario is
carrying an unpaid liability (or an unclaimed refund) for year X, and net worth
flatters whichever scenario owes more.

Stopping at year X in real life means that bill still has to be paid, so the
comparison below deducts each scenario's outstanding balance. It only moves the
answer where the two scenarios owe *different* amounts — which is exactly the 2026
wage year that motivated the question.


## Year-end detail


In [ ]:
def _render_year_end_detail(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return
    nw = net_worth_by_month(result.runs, result.scenarios)
    year_end = year_end_values(nw)
    year_end.index.name = "year"
    for name in result.variants:
        year_end[f"Δ {name}"] = year_end[name] - year_end[result.baseline]
    display(year_end.style.format("${:,.0f}"))


display(widgets.interactive_output(
    _render_year_end_detail, {"baseline": baseline_picker, "variants": variant_picker}
))


## What drives it: Social Security received

Gross benefit (including the Medicare premiums withheld before deposit), so the
two scenarios are compared on the same basis. The cumulative view is the classic
claim-timing picture: the later claim starts lower and crosses over.


In [ ]:
from analysis.sim.scenario_compare import plot_ss_gross, ss_gross_by_month


def _render_ss_gross(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return
    ss = pd.DataFrame(
        {name: ss_gross_by_month(result.runs, name) for name in result.scenarios}
    ).fillna(0).sort_index()
    ss_cumulative = ss.cumsum()
    plot_ss_gross(ss, ss_cumulative, result.colors)

    lifetime_ss = ss_cumulative.iloc[-1]
    display(
        pd.DataFrame({
            "lifetime gross SS": lifetime_ss,
            f"vs {result.baseline}": lifetime_ss - lifetime_ss[result.baseline],
        }).style.format("${:,.0f}")
    )


display(widgets.interactive_output(
    _render_ss_gross, {"baseline": baseline_picker, "variants": variant_picker}
))


## Tax impact

Claiming earlier stacks Social Security on top of whatever else is taxable that
year. In a year with wages, that is expensive twice over: the benefit itself is
taxable *and* the wages push provisional income past the thresholds, so a larger
fraction of the benefit gets taxed (up to 85%).

Liability is attributed to the **tax year it was earned in**; the cash settles the
following April, so a difference shown for year *Y* leaves the bank account in
*Y+1*. Net worth already reflects this — the table below is the explanation, not
an additional effect.


In [ ]:
from analysis.sim.scenario_compare import tax_summary_by_year


def _render_tax_impact(baseline, variants):
    result = render_load_or_warn(DATA_DIR, baseline, variants, MAX_VARIANTS)
    if result is None:
        return

    nw = net_worth_by_month(result.runs, result.scenarios)
    ss_cumulative = pd.DataFrame(
        {name: ss_gross_by_month(result.runs, name) for name in result.scenarios}
    ).fillna(0).sort_index().cumsum()
    lifetime_ss = ss_cumulative.iloc[-1]

    tax = {name: tax_summary_by_year(result.runs, name) for name in result.scenarios}
    tax_delta = pd.DataFrame({
        name: tax[name]["total tax"] - tax[result.baseline]["total tax"] for name in result.variants
    }).fillna(0)

    # Only the years where somebody's tax bill actually moves are worth reading.
    moved = tax_delta[tax_delta.abs().max(axis=1) >= 1]
    display(Markdown(
        f"Tax owed vs `{result.baseline}`, for the {len(moved)} of {len(tax_delta)} years "
        "where it differs (negative = owes less):"
    ))
    display(moved.style.format("${:,.0f}").background_gradient(cmap="RdYlGn_r", axis=None))

    lifetime_tax = pd.Series({name: tax[name]["total tax"].sum() for name in result.scenarios})
    display(
        pd.DataFrame({
            "lifetime tax": lifetime_tax,
            f"vs {result.baseline}": lifetime_tax - lifetime_tax[result.baseline],
            "lifetime gross SS": lifetime_ss,
            "SS net of tax, vs " + result.baseline: (lifetime_ss - lifetime_ss[result.baseline])
            - (lifetime_tax - lifetime_tax[result.baseline]),
            "net worth at end": nw.iloc[-1],
        }).style.format("${:,.0f}")
    )


display(widgets.interactive_output(
    _render_tax_impact, {"baseline": baseline_picker, "variants": variant_picker}
))
